# 11 — Full-data error analysis

Load the final aligned predictions, reproduce both error pools, verify the fixed manual-review samples, and reproduce the BERT-base versus HateBERT A/B/C/D comparison. This notebook never samples or replaces reviewed cases.

In [12]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "data/raw/train.csv").exists()
)
RESULTS = ROOT / "results"
REPRODUCED = ROOT / "reproduced_runs"

FINAL = RESULTS / "error_analysis_full"
TEST = ROOT / "data/splits/full/test.csv"
FULL_PREDICTIONS = FINAL / "all_model_predictions_full.csv"
FIXED_PREDICTIONS = FINAL / "all_model_predictions_fixed_200k.csv"
MODELS = ["LR", "SVM", "DistilBERT", "BERT", "HateBERT"]
EXPECTED_TEST_ROWS = 178083

def load_aligned_predictions(path):
    frame = pd.read_csv(path)
    required = {"id", "text", "true_label"} | {f"{model}_pred" for model in MODELS}
    assert required.issubset(frame.columns), sorted(required - set(frame.columns))
    assert len(frame) == EXPECTED_TEST_ROWS and frame["id"].is_unique
    canonical = pd.read_csv(TEST, usecols=["id", "comment_text", "label"])
    assert set(frame["id"]) == set(canonical["id"])
    aligned = frame.set_index("id").reindex(canonical["id"])
    assert np.array_equal(
        aligned["true_label"].astype("int8").to_numpy(),
        canonical["label"].astype("int8").to_numpy(),
    )
    assert np.array_equal(
        aligned["text"].fillna("").astype(str).to_numpy(),
        canonical["comment_text"].fillna("").astype(str).to_numpy(),
    )
    return aligned.reset_index()

def add_error_flags(frame):
    frame = frame.copy()
    for model in MODELS:
        frame[f"{model}_error"] = (
            frame[f"{model}_pred"].astype("int8") != frame["true_label"].astype("int8")
        )
    return frame

def make_error_pool(frame):
    error_columns = [f"{model}_error" for model in MODELS]
    return frame.loc[frame[error_columns].any(axis=1)].copy()

full_predictions = add_error_flags(load_aligned_predictions(FULL_PREDICTIONS))
fixed_predictions = add_error_flags(load_aligned_predictions(FIXED_PREDICTIONS))
full_error_pool = make_error_pool(full_predictions)
fixed_error_pool = make_error_pool(fixed_predictions)
print("Full-data error pool:", len(full_error_pool))
print("Fixed-200k error pool:", len(fixed_error_pool))
assert len(full_error_pool) == 24154
assert len(fixed_error_pool) == 23795

Full-data error pool: 24154
Fixed-200k error pool: 23795


## Fixed general manual-review sample

The workbook contains 100 general error cases: 79 false-positive cases and 21 false-negative cases. Each comment is counted once, even when several models are wrong.

In [13]:
REVIEW_BOOK = ROOT / "tables/error_analysis_300_organized.xlsx"
general = pd.read_excel(REVIEW_BOOK, sheet_name="General_100")
targeted = pd.read_excel(REVIEW_BOOK, sheet_name="Targeted_200")
group_names = {
    "A": "A_HateBERT_recovers_BERT_FN",
    "B": "B_HateBERT_introduces_FN",
    "C": "C_HateBERT_introduces_FP",
    "D": "D_HateBERT_fixes_BERT_FP",
}
targeted["review_group"] = targeted["group"].map(group_names)
manual_300 = pd.concat([general[["id"]], targeted[["id"]]], ignore_index=True)
assert len(general) == 100 and general["id"].is_unique
assert len(manual_300) == 300 and manual_300["id"].is_unique
assert set(general["id"]).issubset(set(manual_300["id"]))
assert set(general["id"]).issubset(set(full_error_pool["id"]))

general_counts = general["error_type"].value_counts().rename_axis("error_type").reset_index(name="n")
display(general_counts)

,error_type,n
0,FP,79
1,FN,21


,wrong_models,n
0,"BERT, HateBERT",1
1,"DistilBERT, BERT",1
2,"DistilBERT, BERT, HateBERT",7
3,LR,5
4,"LR, DistilBERT, BERT, HateBERT",1
5,"LR, SVM",55
6,"LR, SVM, BERT",1
7,"LR, SVM, DistilBERT",2
8,"LR, SVM, DistilBERT, BERT",1
9,"LR, SVM, DistilBERT, BERT, HateBERT",13


## Recomputed model-error combinations

This table uses the current final predictions. The manual-review workbook remains unchanged.

In [ ]:
general_predictions = full_predictions[full_predictions["id"].isin(general["id"])].copy()
def wrong_models(row):
    return ", ".join(model for model in MODELS if int(row[f"{model}_pred"]) != int(row["true_label"]))
general_predictions["wrong_models"] = general_predictions.apply(wrong_models, axis=1)
display(general_predictions.groupby("wrong_models").size().rename("n").reset_index())

## BERT-base versus HateBERT disagreement pools

A and D are cases where HateBERT corrects BERT-base. B and C are cases where HateBERT introduces an error. The workbook contains 50 reviewed examples from each group.

In [ ]:
groups = {
    "A_HateBERT_recovers_BERT_FN": (
        (full_predictions["true_label"] == 1)
        & (full_predictions["BERT_pred"] == 0)
        & (full_predictions["HateBERT_pred"] == 1)
    ),
    "B_HateBERT_introduces_FN": (
        (full_predictions["true_label"] == 1)
        & (full_predictions["BERT_pred"] == 1)
        & (full_predictions["HateBERT_pred"] == 0)
    ),
    "C_HateBERT_introduces_FP": (
        (full_predictions["true_label"] == 0)
        & (full_predictions["BERT_pred"] == 0)
        & (full_predictions["HateBERT_pred"] == 1)
    ),
    "D_HateBERT_fixes_BERT_FP": (
        (full_predictions["true_label"] == 0)
        & (full_predictions["BERT_pred"] == 1)
        & (full_predictions["HateBERT_pred"] == 0)
    ),
}
group_pools = {name: full_predictions.loc[mask].copy() for name, mask in groups.items()}
group_sizes = {name: len(frame) for name, frame in group_pools.items()}
assert group_sizes == {
    "A_HateBERT_recovers_BERT_FN": 530,
    "B_HateBERT_introduces_FN": 319,
    "C_HateBERT_introduces_FP": 764,
    "D_HateBERT_fixes_BERT_FP": 426,
}

assert len(targeted) == 200 and targeted["id"].is_unique
assert set(manual_300["id"]) == set(general["id"]) | set(targeted["id"])
assert targeted["review_group"].value_counts().to_dict() == {name: 50 for name in groups}
for name, pool in group_pools.items():
    sample_ids = set(targeted.loc[targeted["review_group"] == name, "id"])
    assert sample_ids.issubset(set(pool["id"]))


,group,meaning,pool_n,reviewed_n
0,A,HateBERT corrects a BERT-base false negative,530,50
1,B,HateBERT introduces a false negative,319,50
2,C,HateBERT introduces a false positive,764,50
3,D,HateBERT corrects a BERT-base false positive,426,50


group,A,B,C,D
signal_category,,,,
Direct insult / derogation,14,13,22,17
Group / identity hostility,14,16,13,10
Mild / contextual / no clear toxic signal,4,6,7,11
Sexualised / profanity / stigmatising,10,8,4,7
Violence / threat / harm,8,7,4,5


group,A,B,C,D
function_category,,,,
Descriptive / contextual / self-directed,4,2,1,6
Direct attack / use,14,17,9,10
Group generalisation,4,2,1,1
Metaphor / idiom / hypothetical,4,3,3,5
Other / mixed,5,2,3,0
Quotation / report / counterspeech,4,6,6,6
Sarcasm / irony / rhetorical,8,10,6,8
Topic / opinion criticism,7,8,21,14


## Targeted sample summary

This section summarises the four fixed BERT-base/HateBERT review groups. The 200 rows come from the supplied workbook and are not resampled.


In [ ]:
group_table = pd.DataFrame({
    "group": ["A", "B", "C", "D"],
    "meaning": [
        "HateBERT corrects a BERT-base false negative",
        "HateBERT introduces a false negative",
        "HateBERT introduces a false positive",
        "HateBERT corrects a BERT-base false positive",
    ],
    "pool_n": [group_sizes[name] for name in group_names.values()],
    "reviewed_n": [50, 50, 50, 50],
})
display(group_table)

## Manual classification tables

The following tables use the classification columns in the fixed Targeted-200 sample. They are displayed separately from the pool-size summary.


In [ ]:
classification_columns = ["A", "B", "C", "D"]
signal_table = pd.crosstab(
    targeted["signal_category"],
    targeted["group"],
).reindex(columns=classification_columns, fill_value=0)
function_table = pd.crosstab(
    targeted["function_category"],
    targeted["group"],
).reindex(columns=classification_columns, fill_value=0)
display(signal_table)
display(function_table)

## Continuous toxicity score analysis

In [15]:
assert targeted["continuous_toxicity_score"].between(0, 1).all()
score_summary = (
    targeted.groupby("group")["continuous_toxicity_score"]
    .agg(n="count", mean="mean", median="median", minimum="min", maximum="max")
    .reset_index()
)
targeted["score_band"] = pd.cut(
    targeted["continuous_toxicity_score"],
    bins=[-0.01, 0.49, 0.69, 1.0],
    labels=["below 0.5", "0.5 to 0.69", "0.7 or above"],
)
band_summary = pd.crosstab(targeted["group"], targeted["score_band"])
display(score_summary)
display(band_summary)

for group in ["C", "D"]:
    scores = targeted.loc[
        targeted["group"] == group,
        "continuous_toxicity_score"
    ]
    proportion = ((scores >= 0.30) & (scores < 0.50)).mean()
    print(
        group,
        "median =", scores.median(),
        "proportion_0.30_0.499 =", proportion,
    )

,group,n,mean,median,minimum,maximum
0,A,50,0.637894,0.6,0.5,1.000000
1,B,50,0.614667,0.6,0.5,1.000000
2,C,50,0.238184,0.3,0.0,0.426230
3,D,50,0.245243,0.3,0.0,0.471429


score_band,below 0.5,0.5 to 0.69,0.7 or above
group,,,
A,0,30,20
B,0,34,16
C,50,0,0
D,50,0,0


C median = 0.3 proportion_0.30_0.499 = 0.52
D median = 0.3 proportion_0.30_0.499 = 0.52


## Optional reproduced outputs

Set `WRITE_REPRODUCED_OUTPUTS = True` only when new prediction files are available. The fixed manual samples remain unchanged.

In [16]:
WRITE_REPRODUCED_OUTPUTS = False
if WRITE_REPRODUCED_OUTPUTS:
    output = REPRODUCED / "error_analysis" / "full_data"
    output.mkdir(parents=True, exist_ok=True)
    full_error_pool.to_csv(output / "full_error_pool.csv", index=False)
    fixed_error_pool.to_csv(output / "fixed_200k_error_pool.csv", index=False)
    pd.DataFrame({"group": list(group_sizes), "pool_n": list(group_sizes.values())}).to_csv(
        output / "bert_hatebert_pool_sizes.csv", index=False
    )
    score_summary.to_csv(output / "targeted_score_summary.csv", index=False)